#### Kegiatan 1 — Menyusun Basis Pengetahuan (±50 menit)

In [15]:
aturan = [
    {
        "jika": ["mati total", "lampu indikator mati"],
        "maka": "masalah pada daya",
    },
    {
        "jika": ["masalah pada daya", "baterai kembung"],
        "maka": "DIAGNOSA: baterai rusak, segera ganti",
    },
    {
        "jika": ["masalah pada daya", "charger tidak hangat"],
        "maka": "DIAGNOSA: charger/adaptor rusak",
    },
    {
        "jika": ["menyala", "layar gelap"],
        "maka": "masalah pada layar",
    },
    {
        "jika": ["masalah pada layar", "normal di monitor eksternal"],
        "maka": "DIAGNOSA: kabel fleksibel/panel LCD rusak",
    },
    {
        "jika": ["menyala", "sangat lambat"],
        "maka": "masalah kinerja",
    },
    {
        "jika": ["masalah kinerja", "badan laptop panas"],
        "maka": "DIAGNOSA: overheating, bersihkan kipas & ganti pasta",
    },
    {
        "jika": ["masalah kinerja", "hang saat banyak aplikasi"],
        "maka": "DIAGNOSA: RAM kurang, lakukan upgrade",
    },
]

print("Jumlah aturan:", len(aturan))

Jumlah aturan: 8


Pengembangan 1

In [17]:
semua_kondisi = {kondisi for r in aturan for kondisi in r["jika"]}
kesimpulan_antara = {r["maka"] for r in aturan}
daftar_gejala = sorted(list(semua_kondisi - kesimpulan_antara))

# 2. Tampilkan pilihan ke pengguna
print("=== Sistem Diagnosa Kerusakan Laptop ===")
print("Pilih gejala yang dialami (pisahkan nomor dengan koma, misal: 1, 3):")
for i, g in enumerate(daftar_gejala, 1):
    print(f"{i}. {g}")

pilihan = input("\nNomor gejala: ").split(",")
fakta = {
    daftar_gejala[int(idx.strip()) - 1]
    for idx in pilihan
    if idx.strip().isdigit() and 0 < int(idx.strip()) <= len(daftar_gejala)
}

ada_fakta_baru = True
while ada_fakta_baru:
    ada_fakta_baru = False
    for r in aturan:
        if all(k in fakta for k in r["jika"]) and r["maka"] not in fakta:
            fakta.add(r["maka"])
            ada_fakta_baru = True

hasil = [f for f in fakta if f.startswith("DIAGNOSA:")]

print("\n--- Hasil Analisis ---")
if hasil:
    for h in hasil:
        print(f"-> {h}")
else:
    print("Gejala belum cukup untuk menghasilkan diagnosa pasti.")

=== Sistem Diagnosa Kerusakan Laptop ===
Pilih gejala yang dialami (pisahkan nomor dengan koma, misal: 1, 3):
1. badan laptop panas
2. baterai kembung
3. charger tidak hangat
4. hang saat banyak aplikasi
5. lampu indikator mati
6. layar gelap
7. mati total
8. menyala
9. normal di monitor eksternal
10. sangat lambat

--- Hasil Analisis ---
Gejala belum cukup untuk menghasilkan diagnosa pasti.


#### Kegiatan 2 — Mesin Inferensi: Forward Chaining (±80 menit)

In [18]:
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal) 
    ada_baru = True

    while ada_baru: 
        ada_baru = False
        for r in aturan:
            terpenuhi = all(k in fakta for k in r["jika"])
            if terpenuhi and r["maka"] not in fakta:
                fakta.append(r["maka"])
                print(" Terpicu:", r["jika"], "->", r["maka"])
                ada_baru = True 

    return fakta

gejala = ["mati total", "lampu indikator mati", "baterai kembung"]
hasil = forward_chaining(aturan, gejala)
print("\nKesimpulan akhir:", [f for f in hasil if f.startswith("DIAGNOSA")])

 Terpicu: ['mati total', 'lampu indikator mati'] -> masalah pada daya
 Terpicu: ['masalah pada daya', 'baterai kembung'] -> DIAGNOSA: baterai rusak, segera ganti

Kesimpulan akhir: ['DIAGNOSA: baterai rusak, segera ganti']


Pengembangan 2

In [19]:
def forward_chaining(aturan, fakta_awal, prefix_target="DIAGNOSA"):
    fakta = set(fakta_awal)
    jejak_inferensi = []
    iterasi = 1
    ada_baru = True

    print(f"[*] Fakta awal yang diketahui: {list(fakta)}")
    print("=" * 60)

    while ada_baru:
        ada_baru = False
        print(f"--- Iterasi ke-{iterasi} ---")

        for r in aturan:
            kondisi = r["jika"]
            kesimpulan = r["maka"]

            if all(k in fakta for k in kondisi) and kesimpulan not in fakta:
                fakta.add(kesimpulan)
                ada_baru = True

                log_teks = f"Premis {kondisi} terpenuhi -> Menghasilkan fakta: '{kesimpulan}'"
                jejak_inferensi.append(
                    {"aturan": r, "langkah": iterasi, "log": log_teks}
                )
                print(f"  [+] Terpicu: {log_teks}")

        if not ada_baru:
            print("  (Tidak ada aturan baru yang terpicu)")

        iterasi += 1

    diagnosa = [f for f in fakta if f.startswith(prefix_target)]
    fakta_perantara = [
        f for f in fakta if f not in fakta_awal and not f.startswith(prefix_target)
    ]

    return {
        "semua_fakta": fakta,
        "fakta_perantara": fakta_perantara,
        "diagnosa": diagnosa,
        "riwayat": jejak_inferensi,
    }

gejala = ["mati total", "lampu indikator mati", "baterai kembung"]
hasil = forward_chaining(aturan, gejala)

print("\n" + "=" * 60)
print(f"Fakta Perantara : {hasil['fakta_perantara']}")
print(f"Kesimpulan Akhir: {hasil['diagnosa']}")

[*] Fakta awal yang diketahui: ['baterai kembung', 'lampu indikator mati', 'mati total']
--- Iterasi ke-1 ---
  [+] Terpicu: Premis ['mati total', 'lampu indikator mati'] terpenuhi -> Menghasilkan fakta: 'masalah pada daya'
  [+] Terpicu: Premis ['masalah pada daya', 'baterai kembung'] terpenuhi -> Menghasilkan fakta: 'DIAGNOSA: baterai rusak, segera ganti'
--- Iterasi ke-2 ---
  (Tidak ada aturan baru yang terpicu)

Fakta Perantara : ['masalah pada daya']
Kesimpulan Akhir: ['DIAGNOSA: baterai rusak, segera ganti']


#### Kegiatan 3 — Backward Chaining: Membuktikan Dugaan

In [20]:
def backward_chaining(aturan, fakta, tujuan, level=0):
    spasi = "  " * level 

    if tujuan in fakta:
        print(spasi + f"'{tujuan}' ada di fakta [OK]")
        return True

    for r in aturan:
        if r["maka"] == tujuan:
            print(spasi + f"Cek aturan: {r['jika']} -> {tujuan}")
            if all(
                backward_chaining(aturan, fakta, k, level + 1)
                for k in r["jika"]
            ):
                return True
    return False

fakta = ["mati total", "lampu indikator mati", "baterai kembung"]
dugaan = "DIAGNOSA: baterai rusak, segera ganti"

print(f"Menguji hipotesis: '{dugaan}'\n")
terbukti = backward_chaining(aturan, fakta, dugaan)
print(f"\nTerbukti? {terbukti}")

Menguji hipotesis: 'DIAGNOSA: baterai rusak, segera ganti'

Cek aturan: ['masalah pada daya', 'baterai kembung'] -> DIAGNOSA: baterai rusak, segera ganti
  Cek aturan: ['mati total', 'lampu indikator mati'] -> masalah pada daya
    'mati total' ada di fakta [OK]
    'lampu indikator mati' ada di fakta [OK]
  'baterai kembung' ada di fakta [OK]

Terbukti? True


Pengembangan 3

In [39]:
def backward_chaining(aturan, fakta, tujuan, level=0):
    spasi = "  " * level
    if tujuan in fakta:
        print(spasi + f"[OK] '{tujuan}' ada di fakta")
        return True

    aturan_cocok = [r for r in aturan if r["maka"] == tujuan]
    if aturan_cocok:
        for r in aturan_cocok:
            print(spasi + f"Cek aturan: {r['jika']} -> {tujuan}")
            if all(
                backward_chaining(aturan, fakta, k, level + 1)
                for k in r["jika"]
            ):
                fakta.append(tujuan)
                return True
        return False

    jawab = input(spasi + f"> Apakah laptop '{tujuan}'? (y/n): ").strip().lower()
    if jawab == "y":
        fakta.append(tujuan)
        return True

    return False


fakta = ["mati total"]
dugaan = "DIAGNOSA: charger/adaptor rusak"

print(f"Menguji hipotesis: '{dugaan}'\n")
if backward_chaining(aturan, fakta, dugaan):
    print(f"\nHasil: {dugaan} (TERBUKTI)")
else:
    print(f"\nHasil: Hipotesis tidak terbukti.")

Menguji hipotesis: 'DIAGNOSA: charger/adaptor rusak'


Hasil: Hipotesis tidak terbukti.


#### Kegiatan 4 — Sistem Pakar Interaktif

In [41]:
gejala_dikenal = [
    "mati total",
    "lampu indikator mati",
    "baterai kembung",
    "charger tidak hangat",
    "menyala",
    "layar gelap",
    "normal di monitor eksternal",
    "sangat lambat",
    "badan laptop panas",
    "hang saat banyak aplikasi",
]

print("=== SISTEM PAKAR DIAGNOSA LAPTOP ===")
fakta_user = []
for g in gejala_dikenal:
    jawab = input(f"Apakah laptop Anda '{g}'? (y/t) ")
    if jawab.lower().startswith("y"):
        fakta_user.append(g) 

hasil = forward_chaining(aturan, fakta_user)
diagnosa = [f for f in hasil if f.startswith("DIAGNOSA")]

if diagnosa:
    print("\nHasil pemeriksaan:")
    for d in diagnosa:
        print(" -", d)
else:
    print("\nBelum bisa disimpulkan -- bawa ke teknisi ya!")

=== SISTEM PAKAR DIAGNOSA LAPTOP ===

Hasil pemeriksaan:
 - DIAGNOSA: Charger/adaptor rusak


Pengembangan 4

In [43]:
def forward_chaining(aturan, fakta_awal):
    fakta = list(fakta_awal)
    ada_baru = True
    while ada_baru:
        ada_baru = False
        for r in aturan:
            if all(k in fakta for k in r["jika"]) and r["maka"] not in fakta:
                fakta.append(r["maka"])
                ada_baru = True
    return fakta

print("=== SISTEM PAKAR DIAGNOSA LAPTOP ===")
print("Pilih nomor gejala yang dialami (pisahkan dengan koma, contoh: 1, 2, 4):")
for i, g in enumerate(gejala_dikenal, start=1):
    print(f"[{i:2d}] {g}")

input_raw = input("\nNomor pilihan Anda: ").split(",")

fakta_user = []
for x in input_raw:
    x_bersih = x.strip()
    if x_bersih.isdigit():
        idx = int(x_bersih) - 1
        if 0 <= idx < len(gejala_dikenal):
            gejala = gejala_dikenal[idx]
            if gejala not in fakta_user:
                fakta_user.append(gejala)

print("\n--- GEJALA YANG ANDA PILIH ---")
if fakta_user:
    for idx, f in enumerate(fakta_user, start=1):
        print(f"{idx}. {f}")
else:
    print("Tidak ada nomor gejala valid yang dimasukkan.")

hasil = forward_chaining(aturan, fakta_user)
diagnosa = [f for f in hasil if f.startswith("DIAGNOSA")]

print("\n--- HASIL PEMERIKSAAN ---")
if diagnosa:
    for d in diagnosa:
        print(f"[!] {d}")
else:
    print("Gejala tidak cocok dengan basis aturan -- bawa ke teknisi ya!")

=== SISTEM PAKAR DIAGNOSA LAPTOP ===
Pilih nomor gejala yang dialami (pisahkan dengan koma, contoh: 1, 2, 4):
[ 1] mati total
[ 2] lampu indikator mati
[ 3] baterai kembung
[ 4] charger tidak hangat
[ 5] menyala
[ 6] layar gelap
[ 7] normal di monitor eksternal
[ 8] sangat lambat
[ 9] badan laptop panas
[10] hang saat banyak aplikasi

--- GEJALA YANG ANDA PILIH ---
1. mati total
2. lampu indikator mati
3. charger tidak hangat

--- HASIL PEMERIKSAAN ---
[!] DIAGNOSA: Charger/adaptor rusak
